# Object Tracking Exercise - Module 2

In this exercise, you'll learn to implement object tracking for Physical AI systems. Object tracking is crucial for robots to maintain awareness of objects as they move through the environment.

## Learning Objectives

By the end of this exercise, you will be able to:
1. Understand different object tracking algorithms
2. Implement basic tracking using OpenCV
3. Integrate tracking with ROS 2 for Physical AI systems
4. Evaluate tracking performance in simulated environments

## Prerequisites

Before starting this exercise, ensure you have:
1. Completed the camera integration exercises from Module 1
2. Familiarity with basic computer vision concepts
3. Understanding of ROS 2 message types for images and detections

## Exercise 1: Understanding Object Tracking Concepts

Object tracking involves maintaining the identity and location of objects across multiple frames. Let's start by understanding the basic concepts.

In [ ]:
# Import necessary libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import time

print("Libraries imported successfully")

## Exercise 2: Implementing a Basic Tracker

Let's implement a simple centroid-based tracker that tracks objects by maintaining their center positions.

In [ ]:
class CentroidTracker:
    def __init__(self, max_disappeared=50, max_distance=50):
        self.next_object_id = 0
        self.objects = {}  # Object IDs to centroids
        self.disappeared = {}  # Count frames since object disappeared
        self.max_disappeared = max_disappeared
        self.max_distance = max_distance
    
    def register(self, centroid):
        """Register a new object with the next available ID"""
        self.objects[self.next_object_id] = centroid
        self.disappeared[self.next_object_id] = 0
        self.next_object_id += 1
    
    def deregister(self, object_id):
        """Remove an object ID"""
        del self.objects[object_id]
        del self.disappeared[object_id]
    
    def update(self, rects):
        """Update tracker with new detections"""
        if len(rects) == 0:
            # Mark all existing objects as disappeared
            for object_id in list(self.disappeared.keys()):
                self.disappeared[object_id] += 1
                
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)
            
            return self.objects
        
        # Compute centroids for new detections
        input_centroids = np.zeros((len(rects), 2), dtype="int")
        for (i, (start_x, start_y, end_x, end_y)) in enumerate(rects):
            cx = int((start_x + end_x) / 2.0)
            cy = int((start_y + end_y) / 2.0)
            input_centroids[i] = (cx, cy)
        
        if len(self.objects) == 0:
            # Register all new detections
            for i in range(len(input_centroids)):
                self.register(input_centroids[i])
        else:
            # Match existing objects with new detections
            object_centroids = list(self.objects.values())
            D = np.linalg.norm(np.array(object_centroids)[:, np.newaxis] - input_centroids, axis=2)
            
            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]
            
            used_row_indices = set()
            used_col_indices = set()
            
            for (row, col) in zip(rows, cols):
                if row in used_row_indices or col in used_col_indices:
                    continue
                
                if D[row, col] > self.max_distance:
                    continue
                
                object_id = list(self.objects.keys())[row]
                self.objects[object_id] = input_centroids[col]
                self.disappeared[object_id] = 0
                
                used_row_indices.add(row)
                used_col_indices.add(col)
            
            unused_row_indices = set(range(0, D.shape[0])).difference(used_row_indices)
            unused_col_indices = set(range(0, D.shape[1])).difference(used_col_indices)
            
            if D.shape[0] >= D.shape[1]:
                for row in unused_row_indices:
                    object_id = list(self.objects.keys())[row]
                    self.disappeared[object_id] += 1
                    
                    if self.disappeared[object_id] > self.max_disappeared:
                        self.deregister(object_id)
            else:
                for col in unused_col_indices:
                    self.register(input_centroids[col])
        
        return self.objects

# Create a centroid tracker
tracker = CentroidTracker(max_disappeared=40, max_distance=50)
print("Centroid tracker initialized")

## Exercise 3: Creating a Simple Object Tracking Simulation

Let's create a simple simulation to test our tracker with moving objects.

In [ ]:
# Create a simple simulation environment
def create_simulation_frame(frame_num, object_positions):
    """Create a simulation frame with moving objects"""
    # Create blank frame
    frame = np.zeros((480, 640, 3), dtype="uint8")
    
    # Draw objects at their positions
    for i, pos in enumerate(object_positions):
        # Update position based on frame number
        new_x = int(pos[0] + frame_num * pos[2])  # Add velocity component
        new_y = int(pos[1] + frame_num * pos[3])
        
        # Keep objects within frame bounds
        new_x = max(20, min(620, new_x))
        new_y = max(20, min(460, new_y))
        
        # Draw object
        color = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)][i % 5]
        cv2.circle(frame, (new_x, new_y), 15, color, -1)
        cv2.putText(frame, f'Obj {i}', (new_x-15, new_y-20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    return frame

# Define initial object positions (x, y, vx, vy)
initial_objects = [
    (100, 100, 1, 0.5),    # Object moving diagonally
    (200, 300, -0.8, 1),   # Object moving in opposite direction
    (400, 200, 0.5, -0.7), # Another moving object
]

# Test the tracker with the simulation
def simulate_tracking():
    tracker = CentroidTracker(max_disappeared=40, max_distance=50)
    
    for frame_num in range(100):  # Simulate 100 frames
        # Create simulation frame
        frame = create_simulation_frame(frame_num, initial_objects)
        
        # Detect objects in the frame (simple color-based detection)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        # Detect red, green, blue objects
        lower_red = np.array([0, 50, 50])
        upper_red = np.array([10, 255, 255])
        mask_red = cv2.inRange(hsv, lower_red, upper_red)
        
        lower_green = np.array([40, 50, 50])
        upper_green = np.array([80, 255, 255])
        mask_green = cv2.inRange(hsv, lower_green, upper_green)
        
        lower_blue = np.array([100, 50, 50])
        upper_blue = np.array([130, 255, 255])
        mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)
        
        # Combine masks
        combined_mask = mask_red | mask_green | mask_blue
        
        # Find contours
        contours, _ = cv2.findContours(combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Extract bounding rectangles
        rects = []
        for contour in contours:
            area = cv2.contourArea(contour)
            if area > 100:  # Filter small contours
                x, y, w, h = cv2.boundingRect(contour)
                rects.append((x, y, x+w, y+h))
        
        # Update tracker with detections
        objects = tracker.update(rects)
        
        # Draw tracked objects on frame
        for (object_id, centroid) in objects.items():
            # Draw centroid
            cv2.circle(frame, (int(centroid[0]), int(centroid[1])), 4, (0, 255, 0), -1)
            cv2.putText(frame, f'ID {object_id}', (int(centroid[0])-10, int(centroid[1])-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        
        # Display every 10th frame
        if frame_num % 10 == 0:
            plt.figure(figsize=(10, 6))
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title(f'Frame {frame_num} - Tracked Objects: {len(objects)}')
            plt.axis('off')
            plt.show()
            
        time.sleep(0.1)  # Small delay to simulate real-time processing

# Run the simulation
print("Starting object tracking simulation...")
simulate_tracking()
print("Simulation completed")

## Exercise 4: Implementing Advanced Tracking with OpenCV

Now let's implement a more sophisticated tracker using OpenCV's built-in tracking algorithms.

In [ ]:
# Implement a more advanced tracker using OpenCV's tracking algorithms
class AdvancedObjectTracker:
    def __init__(self):
        self.trackers = cv2.legacy.MultiTracker_create()  # Use legacy for compatibility
        self.object_ids = []
        self.next_id = 0
        
    def add_objects(self, frame, bounding_boxes):
        """Add new objects to track"""
        for bbox in bounding_boxes:
            # Create a single object tracker
            tracker = cv2.legacy.TrackerKCF_create()  # Kernelized Correlation Filters
            
            # Initialize tracker with the bounding box
            tracker.init(frame, tuple(bbox))
            
            # Add to multi-tracker
            self.trackers.add(tracker, frame, tuple(bbox))
            
            # Assign ID
            self.object_ids.append(self.next_id)
            self.next_id += 1
    
    def update(self, frame):
        """Update all trackers with the new frame"""
        success, boxes = self.trackers.update(frame)
        
        tracked_objects = []
        for i, box in enumerate(boxes):
            if success[i]:
                # Convert to int values
                p1 = (int(box[0]), int(box[1]))
                p2 = (int(box[0] + box[2]), int(box[1] + box[3]))
                tracked_objects.append({
                    'id': self.object_ids[i],
                    'bbox': (p1, p2),
                    'center': (int(box[0] + box[2]/2), int(box[1] + box[3]/2))
                })
        
        return tracked_objects

# Create an advanced tracker
advanced_tracker = AdvancedObjectTracker()
print("Advanced tracker initialized")

## Exercise 5: ROS 2 Integration for Object Tracking

Now let's see how to integrate our tracking system with ROS 2 for Physical AI applications.

In [ ]:
# ROS 2 integration example
# This would be part of a ROS 2 node in a real implementation

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image
from vision_msgs.msg import Detection2DArray, Detection2D
from cv_bridge import CvBridge
import cv2
import numpy as np

class TrackingROSNode(Node):
    def __init__(self):
        super().__init__('object_tracking_node')
        
        # Create subscriber for camera images
        self.image_sub = self.create_subscription(
            Image,
            'camera/image_raw',
            self.image_callback,
            10
        )
        
        # Create publisher for tracked objects
        self.tracking_pub = self.create_publisher(
            Detection2DArray,
            'tracked_objects',
            10
        )
        
        # Initialize OpenCV bridge
        self.bridge = CvBridge()
        
        # Initialize tracker
        self.tracker = CentroidTracker(max_disappeared=30, max_distance=40)
        
        self.get_logger().info('Object tracking node initialized')
    
    def image_callback(self, msg):
        """Process incoming image and perform tracking"""
        try:
            # Convert ROS image to OpenCV
            cv_image = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
            
            # Perform object detection (simplified for this example)
            detections = self.simple_object_detection(cv_image)
            
            # Update tracker with detections
            tracked_objects = self.tracker.update(detections)
            
            # Create tracking visualization
            self.visualize_tracking(cv_image, tracked_objects)
            
            # Publish tracking results
            self.publish_tracking_results(tracked_objects, msg.header)
            
        except Exception as e:
            self.get_logger().error(f'Error in image callback: {str(e)}')
    
    def simple_object_detection(self, image):
        """Simple object detection based on color segmentation"""
        # Convert to HSV
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        
        # Define color ranges for different objects
        color_ranges = [
            ([0, 50, 50], [10, 255, 255]),    # Red
            ([40, 50, 50], [80, 255, 255]),   # Green
            ([100, 50, 50], [130, 255, 255]), # Blue
        ]
        
        detections = []
        for lower, upper in color_ranges:
            lower = np.array(lower, dtype="uint8")
            upper = np.array(upper, dtype="uint8")
            mask = cv2.inRange(hsv, lower, upper)
            
            # Find contours
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for contour in contours:
                area = cv2.contourArea(contour)
                if area > 200:  # Filter small detections
                    x, y, w, h = cv2.boundingRect(contour)
                    detections.append((x, y, x+w, y+h))
        
        return detections
    
    def visualize_tracking(self, image, tracked_objects):
        """Visualize tracked objects on the image"""
        for object_id, centroid in tracked_objects.items():
            # Draw centroid
            cv2.circle(image, (int(centroid[0]), int(centroid[1])), 4, (0, 255, 0), -1)
            cv2.putText(image, f'ID {object_id}', (int(centroid[0])-10, int(centroid[1])-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    def publish_tracking_results(self, tracked_objects, header):
        """Publish tracking results as Detection2DArray"""
        detection_array = Detection2DArray()
        detection_array.header = header
        
        for object_id, centroid in tracked_objects.items():
            detection = Detection2D()
            detection.header = header
            
            # Set position (simplified - in a real system, you'd have proper bounding boxes)
            detection.bbox.center.x = float(centroid[0])
            detection.bbox.center.y = float(centroid[1])
            detection.bbox.size_x = 50.0  # Placeholder size
            detection.bbox.size_y = 50.0  # Placeholder size
            
            # Add object ID as a result
            from vision_msgs.msg import ObjectHypothesisWithPose
            hypothesis = ObjectHypothesisWithPose()
            hypothesis.hypothesis.class_id = f'tracked_object_{object_id}'
            hypothesis.hypothesis.score = 0.9  # High confidence for tracked objects
            detection.results.append(hypothesis)
            
            detection_array.detections.append(detection)
        
        self.tracking_pub.publish(detection_array)

# Example of how the node would be used (commented out to avoid ROS issues in notebook)
# def main():
#     rclpy.init()
#     tracking_node = TrackingROSNode()
#     rclpy.spin(tracking_node)
#     tracking_node.destroy_node()
#     rclpy.shutdown()

print("ROS 2 tracking node structure defined")

## Exercise 6: Performance Evaluation

Let's evaluate the performance of our tracking algorithm.

In [ ]:
# Function to evaluate tracking performance
def evaluate_tracking_performance():
    """Evaluate tracking performance metrics"""
    
    # Simulate ground truth and tracked positions
    num_frames = 100
    num_objects = 3
    
    # Ground truth positions (simulated)
    gt_positions = []
    for frame in range(num_frames):
        frame_positions = []
        for obj_id in range(num_objects):
            # Simulate object movement
            x = 100 + obj_id * 150 + frame * 0.5 + np.random.normal(0, 2)
            y = 100 + obj_id * 50 + frame * 0.3 + np.random.normal(0, 2)
            frame_positions.append((x, y))
        gt_positions.append(frame_positions)
    
    # Simulated tracked positions (with some errors)
    tracked_positions = []
    for frame in range(num_frames):
        frame_positions = []
        for obj_id in range(num_objects):
            # Add some tracking error
            gt_x, gt_y = gt_positions[frame][obj_id]
            tracked_x = gt_x + np.random.normal(0, 5)  # 5 pixel average error
            tracked_y = gt_y + np.random.normal(0, 5)
            frame_positions.append((tracked_x, tracked_y))
        tracked_positions.append(frame_positions)
    
    # Calculate performance metrics
    total_error = 0
    valid_detections = 0
    
    for frame in range(num_frames):
        for obj_id in range(num_objects):
            gt_x, gt_y = gt_positions[frame][obj_id]
            tracked_x, tracked_y = tracked_positions[frame][obj_id]
            
            # Calculate Euclidean distance error
            error = np.sqrt((gt_x - tracked_x)**2 + (gt_y - tracked_y)**2)
            total_error += error
            valid_detections += 1
    
    avg_error = total_error / valid_detections if valid_detections > 0 else 0
    
    print(f"Tracking Performance Metrics:")
    print(f"- Average position error: {avg_error:.2f} pixels")
    print(f"- Total tracked positions: {valid_detections}")
    print(f"- Simulation frames: {num_frames}")
    print(f"- Objects tracked: {num_objects}")
    
    # Calculate success rate (assuming < 20 pixels is acceptable)
    acceptable_threshold = 20
    accurate_detections = 0
    
    for frame in range(num_frames):
        for obj_id in range(num_objects):
            gt_x, gt_y = gt_positions[frame][obj_id]
            tracked_x, tracked_y = tracked_positions[frame][obj_id]
            
            error = np.sqrt((gt_x - tracked_x)**2 + (gt_y - tracked_y)**2)
            if error < acceptable_threshold:
                accurate_detections += 1
    
    success_rate = (accurate_detections / valid_detections) * 100 if valid_detections > 0 else 0
    print(f"- Success rate (< {acceptable_threshold}px error): {success_rate:.2f}%")
    
    return avg_error, success_rate

# Evaluate performance
avg_error, success_rate = evaluate_tracking_performance()

## Exercise 7: Challenge - Implement Your Own Tracker

Now it's your turn to implement a custom tracking algorithm. Try to improve upon the centroid tracker by implementing a Kalman filter-based tracker.

In [ ]:
# TODO: Implement a Kalman filter based tracker
# Hint: Use OpenCV's KalmanFilter class or implement your own

class KalmanTracker:
    def __init__(self, dt=1, process_noise=1e-2, measurement_noise=1e-1):
        # Initialize Kalman filter for 2D tracking (x, y, vx, vy)
        self.kalman = cv2.KalmanFilter(4, 2)  # 4 state vars (x,y,vx,vy), 2 measurement vars (x,y)
        
        # State transition matrix
        self.kalman.transitionMatrix = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ], dtype=np.float32)
        
        # Measurement matrix
        self.kalman.measurementMatrix = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ], dtype=np.float32)
        
        # Process noise
        self.kalman.processNoiseCov = np.eye(4, dtype=np.float32) * process_noise
        
        # Measurement noise
        self.kalman.measurementNoiseCov = np.eye(2, dtype=np.float32) * measurement_noise
        
        # Error covariance
        self.kalman.errorCovPost = np.eye(4, dtype=np.float32)
        
        self.initialized = False
        self.prediction = np.zeros((4, 1), dtype=np.float32)
    
    def update(self, measurement):
        """Update the tracker with a new measurement"""
        measurement = np.array([[np.float32(measurement[0])], [np.float32(measurement[1])]])
        
        if not self.initialized:
            # Initialize state
            self.kalman.statePre = np.array([
                [measurement[0]],
                [measurement[1]],
                [0],
                [0]
            ], dtype=np.float32)
            self.initialized = True
            return measurement.flatten()
        
        # Correct the state with the measurement
        self.kalman.correct(measurement)
        
        # Predict next state
        self.prediction = self.kalman.predict()
        
        return self.prediction[:2].flatten()  # Return predicted position (x, y)

# Test the Kalman tracker
kalman_tracker = KalmanTracker()
print("Kalman tracker initialized")

# Simulate tracking with Kalman filter
measurements = [(100 + i*2, 100 + i*1.5) for i in range(20)]  # Simulated measurements
predictions = []

for measurement in measurements:
    prediction = kalman_tracker.update(measurement)
    predictions.append(prediction)

print(f"Kalman tracker processed {len(measurements)} measurements")
print(f"Final predicted position: ({predictions[-1][0]:.2f}, {predictions[-1][1]:.2f})")

## Summary

In this exercise, you learned:
1. How to implement basic object tracking using centroid-based methods
2. How to create more advanced trackers using OpenCV's built-in algorithms
3. How to integrate tracking systems with ROS 2 for Physical AI applications
4. How to evaluate tracking performance using appropriate metrics
5. How to implement a Kalman filter-based tracker for improved accuracy

Object tracking is essential for Physical AI systems as it enables robots to maintain awareness of objects in their environment over time, which is crucial for tasks like navigation, manipulation, and human-robot interaction.

## Next Steps

After completing this exercise, you should:
1. Experiment with different tracking algorithms (KCF, MOSSE, CSRT)
2. Try implementing tracking with deep learning-based detectors
3. Explore multi-object tracking with data association techniques
4. Move on to the next module which covers cognition and control systems